In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred))
    )


def mase(y_true, y_pred, y_train):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)

    naive_error = np.mean(np.abs(np.diff(y_train)))
    return np.mean(np.abs(y_true - y_pred)) / (naive_error)

def theil_u(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denominator = np.sqrt(np.mean(y_true ** 2) + np.mean(y_pred ** 2))
    u = rmse / denominator
    return u

# Loading the data
df = pd.read_excel('output_week.xlsx')

# Counting values and sorting by week
data = df['WEEK'].value_counts().reset_index()
data.columns = ['WEEK', 'DATA']

# Converting week to a date representing the start of the week
data['WEEK_period'] = pd.to_datetime(data['WEEK'] + '-1', format='%G-%V-%u', errors='coerce')
data = data.sort_values('WEEK_period').reset_index(drop=True)

# ------------------------------
# Smoothing (moving average)
# ------------------------------
data['DATA_smooth'] = data['DATA'].rolling(window=5, center=True).mean()
data['DATA_smooth'].fillna(method='bfill', inplace=True)
data['DATA_smooth'].fillna(method='ffill', inplace=True)

# Log transformation of the data
data['y'] = np.log1p(data[['DATA_smooth']])

/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1894/2855273452.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1894/2855273452.py:46: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1894/285527345

In [5]:
# ------------------------------
# Time Series Cross Validation
# ------------------------------

tscv = TimeSeriesSplit(n_splits=10)

rmse_hw = []
mae_hw = []
mase_hw = []
smape_hw = []
folds_validos = []

SEASONAL_PERIOD = 53


for fold, (train_idx, test_idx) in enumerate(tscv.split(data["y"]), 1):

    print(f"\nFold {fold}")

    # ------------------------------
    # Split temporal
    # ------------------------------

    y_train = data["y"].iloc[train_idx]
    y_test = data["y"].iloc[test_idx]


    # ------------------------------
    # Definir sazonalidade
    # ------------------------------

    if len(y_train) >= 2 * SEASONAL_PERIOD:

        seasonal = "add"
        seasonal_periods = SEASONAL_PERIOD

        print(
            f"Fold {fold}: Holt-Winters sazonal"
        )

    else:

        seasonal = None
        seasonal_periods = None

        print(
            f"Fold {fold}: Holt sem sazonalidade "
            f"(treino={len(y_train)})"
        )


    # ------------------------------
    # Holt-Winters
    # ------------------------------

    model_hw = ExponentialSmoothing(
        y_train,
        trend="add",
        seasonal=seasonal,
        seasonal_periods=seasonal_periods,
        initialization_method="estimated"
    ).fit(
        optimized=True
    )


    # ------------------------------
    # Forecast
    # ------------------------------

    y_pred = model_hw.forecast(
        steps=len(y_test)
    )


    # ------------------------------
    # Escala original
    # ------------------------------

    y_train_real = np.expm1(
        y_train.to_numpy()
    )

    y_test_real = np.expm1(
        y_test.to_numpy()
    )

    y_pred_real = np.expm1(
        np.asarray(y_pred)
    )


    # ------------------------------
    # Métricas
    # ------------------------------

    rmse_value = np.sqrt(
        mean_squared_error(
            y_test_real,
            y_pred_real
        )
    )


    mae_value = mean_absolute_error(
        y_test_real,
        y_pred_real
    )


    mase_value = mase(
        y_test_real,
        y_pred_real,
        y_train_real
    )


    smape_value = smape(
        y_test_real,
        y_pred_real
    )


    # ------------------------------
    # Guardar resultados
    # ------------------------------

    folds_validos.append(fold)

    rmse_hw.append(rmse_value)
    mae_hw.append(mae_value)
    mase_hw.append(mase_value)
    smape_hw.append(smape_value)


    print(
        f"RMSE={rmse_value:.4f} | "
        f"MAE={mae_value:.4f} | "
        f"MASE={mase_value:.4f} | "
        f"SMAPE={smape_value:.4f}"
    )


# ------------------------------
# Salvar resultados
# ------------------------------

results_hw = pd.DataFrame({

    "Fold": folds_validos,
    "RMSE": rmse_hw,
    "MAE": mae_hw,
    "MASE": mase_hw,
    "SMAPE": smape_hw

})


results_hw.to_csv(
    "HoltWinters_results.csv",
    index=False
)


Fold 1
Fold 1: Holt sem sazonalidade (treino=40)
RMSE=8.9660 | MAE=7.5261 | MASE=4.8757 | SMAPE=37.6945

Fold 2
Fold 2: Holt sem sazonalidade (treino=76)
RMSE=8.7589 | MAE=7.6389 | MASE=4.8635 | SMAPE=56.3689

Fold 3
Fold 3: Holt-Winters sazonal
RMSE=2.9648 | MAE=2.4493 | MASE=1.7056 | SMAPE=20.5618

Fold 4
Fold 4: Holt-Winters sazonal
RMSE=1.4880 | MAE=1.2565 | MASE=0.9180 | SMAPE=13.7812

Fold 5
Fold 5: Holt-Winters sazonal
RMSE=2.6081 | MAE=2.2519 | MASE=1.7794 | SMAPE=27.3635

Fold 6
Fold 6: Holt-Winters sazonal
RMSE=3.5190 | MAE=2.8768 | MASE=2.4176 | SMAPE=32.8322

Fold 7
Fold 7: Holt-Winters sazonal
RMSE=3.6573 | MAE=2.9772 | MASE=2.6527 | SMAPE=35.4075

Fold 8
Fold 8: Holt-Winters sazonal
RMSE=3.9372 | MAE=3.5300 | MASE=3.2945 | SMAPE=29.4465

Fold 9
Fold 9: Holt-Winters sazonal
RMSE=4.5955 | MAE=4.1766 | MASE=4.0455 | SMAPE=41.0187

Fold 10
Fold 10: Holt-Winters sazonal
RMSE=0.9705 | MAE=0.7593 | MASE=0.7473 | SMAPE=11.7376
